### Packages

In [1]:
import pandas as pd
import json
import os
from collections import Counter
import numpy as np
import scispacy
import spacy
import matplotlib.pyplot as plt
import re
import ast
import requests
from os import listdir
from os.path import isfile, join
from functools import reduce

In [2]:
data_dir = os.getcwd()
data_dir

'/home/eidf128/eidf128/shared/export/juliana/export/juliana'

In [3]:
df_IDR = pd.read_csv("df_IDR_collapse_20260406.csv")
df_IDR

,Unnamed: 0,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Key_Words,Study_Organism,Study_Organism_Term_Source_REF,...,secondobjectnumber,Phenotype_Annotation_Level,Study_Description_clean,Experiment_Description_clean,Study_Description_word_count_clean,Experiment_Description_word_count_clean,Study_Description_clean_lemmatize,Study_Description_clean_entities,Experiment_Description_clean_lemmatize,Experiment_Description_clean_entities
0,0,idr0092,a semi-automated organoid screening method dem...,compound library screen,efo,efo_0007553,intestinal organoids are an excellent model to...,organoids,mus musculus,ncbitaxon,...,NaN,NaN,"['intestinal', 'organoids', 'excellent', 'mode...",NaN,89,0,intestinal organoid excellent model study epit...,"[intestinal, model study, epithelial biology, ...",NaN,[]
1,1,idr0016,human u2os cells - compound cell-painting expe...,high content screen,efo,efo_0007550,phenotypic profiling attempts to summarize mul...,NaN,homo sapiens,ncbitaxon,...,NaN,NaN,"['phenotypic', 'profiling', 'attempts', 'summa...",NaN,182,0,phenotypic profiling attempt summarize multipa...,"[phenotypic profiling, multiparametric analysi...",NaN,[]
2,2,idr0079,an image-based data-driven analysis of cellula...,microscopy assay,efo,efo_0002909,a data-driven analysis of cell morphology and ...,image analysis,danio rerio,ncbitaxon,...,NaN,NaN,"['analysis', 'cell', 'morphology', 'intracellu...","['confocal', 'imaging', 'fixed', 'samples', 'z...",60,20,analysis cell morphology intracellular organiz...,"[morphology, intracellular, organization, zebr...",confocal imaging fixed sample zebrafish poster...,"[confocal imaging, posterior lateral line, pri..."
3,3,idr0067,meiotic cellular rejuvenation is coupled to nu...,time-lapse imaging,omit,omit_0027490,production of healthy gametes in meiosis relie...,meiosis,saccharomyces cerevisiae\n,ncbitaxon,...,NaN,NaN,"['production', 'healthy', 'gametes', 'meiosis'...","['fixed', 'cell', 'fluorescence', 'microscopy'...",99,29,production healthy gamete meiosis rely quality...,"[production, healthy, meiosis, quality, distri...",fix cell fluorescence microscopy transmission ...,"[cell fluorescence microscopy, image, yeast, m..."
4,4,idr0134,reference bioimaging dataset to assess the phe...,histology,none,none,"here, we present a high-quality reference data...",phenotypes,diplophyllum albicans,ncbitaxon,...,NaN,NaN,"['present', 'reference', 'dataset', 'containin...","['representative', 'voucher', 'specimens', 're...",32,508,present reference dataset contain macroscopic ...,"[dataset, macroscopic, phenotypic property spe...",representative voucher specimen receive herbar...,"[specimen, herbaria, sample diplophyllum taxif..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,127,idr0023,nuclear pore scaffold structure analyzed by su...,protein localization,efo,go:0008104,much of life's essential molecular machinery c...,NaN,homo sapiens,ncbitaxon,...,NaN,NaN,"['much', 'lifes', 'essential', 'molecular', 'm...","['systematic', 'immunolabelling', 'complex', '...",75,11,much lifes essential molecular machinery consi...,"[lifes, molecular machinery, protein assembly,...",systematic immunolabelling complex monomeric e...,"[systematic immunolabelling, complex, monomeri..."
128,128,idr0038,ex vivo live cell tracking in kidney organoids...,time-lapse imaging,omit,omit_0027490,we have adapted the mouse kidney rudiment assa...,NaN,mus musculus,ncbitaxon,...,NaN,NaN,"['adapted', 'mouse', 'kidney', 'rudiment', 'as...","['adapted', 'mouse', 'kidney', 'rudiment', 'as...",79,161,adapt mouse kidney rudiment assay generate ren...,"[adapt, mouse kidney rudiment assay, renal org...",adapt mouse kidney rudiment assay generate ren...,"[adapt, mouse kidney rudiment assay, renal org..."
129,129,idr0138,integration of spatial and single-cell transcr...,seqfish,efo,efo_0008991,molecular profiling of single cell

### Models

In [47]:
import torch
import tqdm as notebook_tqdm
#from transformers import GenerationConfig

In [48]:
from transformers import pipeline
model_id = "meta-llama/Llama-3.2-3B-Instruct"
#model_id = "mistralai/Mistral-7B-Instruct-v0.3"

In [49]:
pipe = pipeline("text-generation",
                model = model_id,
                #torch_dtype = torch.bfloat16, ## Use this with Mistral
                torch_dtype = torch.bfloat16,
                device_map="auto",
                pad_token_id=128001 
               )

Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████| 2/2 [00:01<00:00,  1.51it/s]
Device set to use cuda:0


In [17]:
pipe.model.config.max_length = None
pipe.model.generation_config.max_length = None  # clear both places

In [7]:
message = [
    {"role": "user", "content": "Who are you? Please, answer in pirate-speak."},
]

In [8]:
outputs = pipe(message)

In [9]:
response = outputs[0]["generated_text"][-1]["content"]
print(response)

Yer lookin' fer a tale o' who I be, eh? Alright then, mate


#### Test with one entry

In [10]:
test = df_IDR

In [11]:
test["Study_Description"]

0      intestinal organoids are an excellent model to...
1      phenotypic profiling attempts to summarize mul...
2      a data-driven analysis of cell morphology and ...
3      production of healthy gametes in meiosis relie...
4      here, we present a high-quality reference data...
                             ...                        
127    much of life's essential molecular machinery c...
128    we have adapted the mouse kidney rudiment assa...
129    molecular profiling of single cells has advanc...
130    viral infectious diseases span a myriad of mal...
131    we describe a dataset obtained by applying our...
Name: Study_Description, Length: 132, dtype: str

In [12]:
test["Description_combined"] = (
    test[["Study_Description", "Experiment_Description", "Protocol_Description"]]
    .fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [13]:
test

,Unnamed: 0,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Key_Words,Study_Organism,Study_Organism_Term_Source_REF,...,Phenotype_Annotation_Level,Study_Description_clean,Experiment_Description_clean,Study_Description_word_count_clean,Experiment_Description_word_count_clean,Study_Description_clean_lemmatize,Study_Description_clean_entities,Experiment_Description_clean_lemmatize,Experiment_Description_clean_entities,Description_combined
0,0,idr0092,a semi-automated organoid screening method dem...,compound library screen,efo,efo_0007553,intestinal organoids are an excellent model to...,organoids,mus musculus,ncbitaxon,...,NaN,"['intestinal', 'organoids', 'excellent', 'mode...",NaN,89,0,intestinal organoid excellent model study epit...,"[intestinal, model study, epithelial biology, ...",NaN,[],intestinal organoids are an excellent model to...
1,1,idr0016,human u2os cells - compound cell-painting expe...,high content screen,efo,efo_0007550,phenotypic profiling attempts to summarize mul...,NaN,homo sapiens,ncbitaxon,...,NaN,"['phenotypic', 'profiling', 'attempts', 'summa...",NaN,182,0,phenotypic profiling attempt summarize multipa...,"[phenotypic profiling, multiparametric analysi...",NaN,[],phenotypic profiling attempts to summarize mul...
2,2,idr0079,an image-based data-driven analysis of cellula...,microscopy assay,efo,efo_0002909,a data-driven analysis of cell morphology and ...,image analysis,danio rerio,ncbitaxon,...,NaN,"['analysis', 'cell', 'morphology', 'intracellu...","['confocal', 'imaging', 'fixed', 'samples', 'z...",60,20,analysis cell morphology intracellular organiz...,"[morphology, intracellular, organization, zebr...",confocal imaging fixed sample zebrafish poster...,"[confocal imaging, posterior lateral line, pri...",a data-driven analysis of cell morphology and ...
3,3,idr0067,meiotic cellular rejuvenation is coupled to nu...,time-lapse imaging,omit,omit_0027490,production of healthy gametes in meiosis relie...,meiosis,saccharomyces cerevisiae\n,ncbitaxon,...,NaN,"['production', 'healthy', 'gametes', 'meiosis'...","['fixed', 'cell', 'fluorescence', 'microscopy'...",99,29,production healthy gamete meiosis rely quality...,"[production, healthy, meiosis, quality, distri...",fix cell fluorescence microscopy transmission ...,"[cell fluorescence microscopy, image, yeast, m...",production of healthy gametes in meiosis relie...
4,4,idr0134,reference bioimaging dataset to assess the phe...,histology,none,none,"here, we present a high-quality reference data...",phenotypes,diplophyllum albicans,ncbitaxon,...,NaN,"['present', 'reference', 'dataset', 'containin...","['representative', 'voucher', 'specimens', 're...",32,508,present reference dataset contain macroscopic ...,"[dataset, macroscopic, phenotypic property spe...",representative voucher specimen receive herbar...,"[specimen, herbaria, sample diplophyllum taxif...","here, we present a high-quality reference data..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,127,idr0023,nuclear pore scaffold structure analyzed by su...,protein localization,efo,go:0008104,much of life's essential molecular machinery c...,NaN,homo sapiens,ncbitaxon,...,NaN,"['much', 'lifes', 'essential', 'molecular', 'm...","['systematic', 'immunolabelling', 'complex', '...",75,11,much lifes essential molecular machinery consi...,"[lifes, molecular machinery, protein assembly,...",systematic immunolabelling complex monomeric e...,"[systematic immunolabelling, complex, monomeri...",much of life's essential molecular machinery c...
128,128,idr0038,ex vivo live cell tracking in kidney organoids...,time-lapse imaging,omit,omit_0027490,we have adapted the mouse kidney rudiment assa...,NaN,mus musculus,ncbitaxon,...,NaN,"['adapted', 'mouse', 'kidney', 'rudiment', 'as...","['adapted', 'mouse', 'kidney', 'rudiment', 'as...",79,161,adapt mouse kidney rudiment assay generate ren...,"[

#### Only descriptions

##### Species:

In [66]:
start_time = time.time()
for idx, row in test.iterrows():
    message = [
    {"role": "system", "content": "This is your context: " + test.at[idx,"Description_combined"]},
    {"role": "user", "content": "Based on the context provided, give me the name of the scientific species that could be used in this study. If you don't know, say, I don't know."},
     ]
    outputs = pipe(message, max_new_tokens = 256)
    response = outputs[0]["generated_text"][-1]["content"]
    
    test.at[idx, "Llama_description_combined_species"] = response

    print(f'The row {idx} has been proceed.')
    
end_time = time.time()

print(f"Execution time: {end_time - start_time:.2f} seconds")


The row 0 has been proceed.
The row 1 has been proceed.
The row 2 has been proceed.
The row 3 has been proceed.
The row 4 has been proceed.
The row 5 has been proceed.
The row 6 has been proceed.
The row 7 has been proceed.
The row 8 has been proceed.
The row 9 has been proceed.
The row 10 has been proceed.
The row 11 has been proceed.
The row 12 has been proceed.
The row 13 has been proceed.
The row 14 has been proceed.
The row 15 has been proceed.
The row 16 has been proceed.
The row 17 has been proceed.
The row 18 has been proceed.
The row 19 has been proceed.
The row 20 has been proceed.
The row 21 has been proceed.
The row 22 has been proceed.
The row 23 has been proceed.
The row 24 has been proceed.
The row 25 has been proceed.
The row 26 has been proceed.
The row 27 has been proceed.
The row 28 has been proceed.
The row 29 has been proceed.
The row 30 has been proceed.
The row 31 has been proceed.
The row 32 has been proceed.
The row 33 has been proceed.
The row 34 has been proc

In [67]:
df_IDR

,Unnamed: 0,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Key_Words,Study_Organism,Study_Organism_Term_Source_REF,...,Experiment_Description_clean,Study_Description_word_count_clean,Experiment_Description_word_count_clean,Study_Description_clean_lemmatize,Study_Description_clean_entities,Experiment_Description_clean_lemmatize,Experiment_Description_clean_entities,Description_combined,Llama_description_combined_species,Real_Species_in_Response_descriptions
0,0,idr0092,a semi-automated organoid screening method dem...,compound library screen,efo,efo_0007553,intestinal organoids are an excellent model to...,organoids,mus musculus,ncbitaxon,...,NaN,89,0,intestinal organoid excellent model study epit...,"[intestinal, model study, epithelial biology, ...",NaN,[],intestinal organoids are an excellent model to...,"Based on the context provided, the scientific ...",No
1,1,idr0016,human u2os cells - compound cell-painting expe...,high content screen,efo,efo_0007550,phenotypic profiling attempts to summarize mul...,NaN,homo sapiens,ncbitaxon,...,NaN,182,0,phenotypic profiling attempt summarize multipa...,"[phenotypic profiling, multiparametric analysi...",NaN,[],phenotypic profiling attempts to summarize mul...,"Based on the context provided, the scientific ...",No
2,2,idr0079,an image-based data-driven analysis of cellula...,microscopy assay,efo,efo_0002909,a data-driven analysis of cell morphology and ...,image analysis,danio rerio,ncbitaxon,...,"['confocal', 'imaging', 'fixed', 'samples', 'z...",60,20,analysis cell morphology intracellular organiz...,"[morphology, intracellular, organization, zebr...",confocal imaging fixed sample zebrafish poster...,"[confocal imaging, posterior lateral line, pri...",a data-driven analysis of cell morphology and ...,"Based on the context provided, the scientific ...",No
3,3,idr0067,meiotic cellular rejuvenation is coupled to nu...,time-lapse imaging,omit,omit_0027490,production of healthy gametes in meiosis relie...,meiosis,saccharomyces cerevisiae\n,ncbitaxon,...,"['fixed', 'cell', 'fluorescence', 'microscopy'...",99,29,production healthy gamete meiosis rely quality...,"[production, healthy, meiosis, quality, distri...",fix cell fluorescence microscopy transmission ...,"[cell fluorescence microscopy, image, yeast, m...",production of healthy gametes in meiosis relie...,"Based on the context provided, the scientific ...",No
4,4,idr0134,reference bioimaging dataset to assess the phe...,histology,none,none,"here, we present a high-quality reference data...",phenotypes,diplophyllum albicans,ncbitaxon,...,"['representative', 'voucher', 'specimens', 're...",32,508,present reference dataset contain macroscopic ...,"[dataset, macroscopic, phenotypic property spe...",representative voucher specimen receive herbar...,"[specimen, herbaria, sample diplophyllum taxif...","here, we present a high-quality reference data...","Based on the context provided, the scientific ...",No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,127,idr0023,nuclear pore scaffold structure analyzed by su...,protein localization,efo,go:0008104,much of life's essential molecular machinery c...,NaN,homo sapiens,ncbitaxon,...,"['systematic', 'immunolabelling', 'complex', '...",75,11,much lifes essential molecular machinery consi...,"[lifes, molecular machinery, protein assembly,...",systematic immunolabelling complex monomeric e...,"[systematic immunolabelling, complex, monomeri...",much of life's essential molecular machinery c...,"Based on the context provided, the scientific ...",No
128,128,idr0038,ex vivo live cell tracking in kidney organoids...,time-lapse imaging,omit,omit_0027490,we have adapted the mouse kidney rudiment assa...,NaN,mus musculus,ncbitaxon,...,"['adapted', 'mouse', 'kidney', 'rudiment', 'as...",79,161,adapt mouse kidney rudiment assay generate ren...,"[adapt, mouse kidney rudiment assay, rena

In [68]:
def remove_special_characters(text):
    """
    This function removes special characters from the text.
    :param text: text
    :return: text without special characters
    """
    cleaned_string = re.sub(r'[?|$|.|!|@|#|%|^|&|*|(|)|-|_|+|=|;|:|,|<|>|/|{|}|[|]|~|`|\'|\"|\\]',r'', text)

    return cleaned_string

In [69]:
df_IDR.at[3,'Study_Organism']

'saccharomyces cerevisiae\n'

In [70]:
df_IDR.at[3,'Llama_description_combined_species']

"Based on the context provided, the scientific species that could be used in this study is Saccharomyces cerevisiae, also known as baker's yeast or budding yeast."

In [71]:
count_match = 0
for row, i in test.iterrows():
    
    if isinstance(i['Study_Organism'], str) and i['Study_Organism'].strip() != '':

        real_specie = i['Study_Organism']
        response = i['Llama_description_combined_species']
        clean_response = remove_special_characters(response)
        lower_clean_response = clean_response.lower()

        if real_specie in lower_clean_response:
            print('The real species is in the response.')
            test.at[row, 'Real_Species_in_Response_descriptions'] = 'Yes'
            count_match +=1
       
        else:
            print('The real species is NOT in the response.')
            test.at[row, 'Real_Species_in_Response_descriptions'] = 'No'
    else:
        print('No real species to check in the response.')
        test.at[row, 'Real_Species_in_Response_descriptions'] = 'No species provided'

The real species is in the response.
The real species is NOT in the response.
The real species is in the response.
The real species is NOT in the response.
The real species is NOT in the response.
The real species is in the response.
The real species is in the response.
The real species is NOT in the response.
The real species is in the response.
The real species is NOT in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is NOT in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is NOT in the response.
The real species is NOT in the response.
The real species is in the response.
The real species is in the response.
The real species is in the response.
The real species is NOT in the response.
The real species is in the response.
The real species is in the response.
The real species is NOT in the response

In [72]:
count_match 

69

##### Study type:

In [21]:
test.columns

Index(['Unnamed: 0', 'Comment[IDR_Study_Accession]', 'Study_Title',
       'Study_Type', 'Study_Type_Term_Source_REF', 'Study_Type_Term_Accession',
       'Study_Description', 'Study_Key_Words', 'Study_Organism',
       'Study_Organism_Term_Source_REF',
       ...
       'Experiment_Description_clean', 'Study_Description_word_count_clean',
       'Experiment_Description_word_count_clean',
       'Study_Description_clean_lemmatize', 'Study_Description_clean_entities',
       'Experiment_Description_clean_lemmatize',
       'Experiment_Description_clean_entities', 'Description_combined',
       'Mistral_description_combined_species',
       'Real_Species_in_Response_descriptions'],
      dtype='str', length=234)

In [22]:
raw_experiment_types = test['Study_Type'].to_list()
raw_experiment_types

['compound library screen',
 'high content screen',
 'microscopy assay',
 'time-lapse imaging',
 'histology',
 'time-lapse imaging',
 'high content screen',
 'multiplexed immunofluorescence',
 'high content screen',
 'protein localization',
 'time-lapse imaging',
 'electron microscopy volume map',
 'high content screen',
 'time-lapse imaging',
 'high content screen',
 'high content screen',
 'high content screen',
 'in situ sequencing\n',
 'protein localization',
 'high content screen',
 'time-lapse imaging',
 'time-lapse imaging',
 'high content screen',
 'electron microscopy volume map',
 'time-lapse imaging',
 'protein localization ',
 'high content screen',
 'in-situ hybridization assay',
 'electron microscopy volume map',
 'spindle assembly\n',
 'high content screen',
 'protein localization',
 'high content screen\n',
 'time-lapse imaging\n',
 'high content screen',
 'high content screen',
 'high content screen',
 'high content screen',
 'high content screen',
 'time-lapse imaging

In [23]:
# Make a list with all types of experiments
# This list will be used to create a list of all types of experiments, removing duplicates and cleaning the data.
# ------------------------------------------------------------
list_experiment_types_clean = []
for element in raw_experiment_types:
    element_clean = element.strip().replace('\n', '')
    list_experiment_types_clean.append(element_clean)

In [24]:
experiment_types = list(set(list_experiment_types_clean))
experiment_types.sort()
len(experiment_types)

30

In [25]:
experiment_types

['compound library screen',
 'dna sequencing',
 'electron microscopy volume map',
 'fluorescence in situ hybridization',
 'high content analysis of cells',
 'high content screen',
 'high content screen of cells treated with a compound library',
 'histology',
 'image cytometry',
 'image segmentation',
 'imaging method',
 'immunocytochemistry',
 'in situ sequencing',
 'in-situ hybridization assay',
 'infection',
 'machine learning',
 'metabolic network measurement',
 'micrograph',
 'microscopy assay',
 'morphogenesis',
 'multiplexed immunofluorescence',
 'myelination',
 'phenotype',
 'process of establishing viral infection',
 'protein localization',
 'response to cold',
 'seqfish',
 'spindle assembly',
 'time-lapse imaging',
 'x-chromosome inactivation']

In [26]:
', '.join(experiment_types)

'compound library screen, dna sequencing, electron microscopy volume map, fluorescence in situ hybridization, high content analysis of cells, high content screen, high content screen of cells treated with a compound library, histology, image cytometry, image segmentation, imaging method, immunocytochemistry, in situ sequencing, in-situ hybridization assay, infection, machine learning, metabolic network measurement, micrograph, microscopy assay, morphogenesis, multiplexed immunofluorescence, myelination, phenotype, process of establishing viral infection, protein localization, response to cold, seqfish, spindle assembly, time-lapse imaging, x-chromosome inactivation'

In [27]:
for idx, row in test.iterrows():
    message = [
    {"role": "system", "content": i['Study_Description']},
    {"role": "user", "content": "Choose the experiment type for this study from the following list: " + ', '.join(experiment_types) + ". If you don't know, just say 'I don't know'."},
     ]
    outputs = pipe(message, max_new_tokens = 256)
    response = outputs[0]["generated_text"][-1]["content"]
    
    test.at[idx, "Mistral_description_study_type"] = response

    print(f'The row {idx} has been proceed.')

The row 0 has been proceed.
The row 1 has been proceed.
The row 2 has been proceed.
The row 3 has been proceed.
The row 4 has been proceed.
The row 5 has been proceed.
The row 6 has been proceed.
The row 7 has been proceed.
The row 8 has been proceed.
The row 9 has been proceed.
The row 10 has been proceed.
The row 11 has been proceed.
The row 12 has been proceed.
The row 13 has been proceed.
The row 14 has been proceed.
The row 15 has been proceed.
The row 16 has been proceed.
The row 17 has been proceed.
The row 18 has been proceed.
The row 19 has been proceed.
The row 20 has been proceed.
The row 21 has been proceed.
The row 22 has been proceed.
The row 23 has been proceed.
The row 24 has been proceed.
The row 25 has been proceed.
The row 26 has been proceed.
The row 27 has been proceed.
The row 28 has been proceed.
The row 29 has been proceed.
The row 30 has been proceed.
The row 31 has been proceed.
The row 32 has been proceed.
The row 33 has been proceed.
The row 34 has been proc

In [28]:
df_IDR['Study_Type'][4]

'histology'

In [29]:
df_IDR['Mistral_description_study_type'][4]

" The experiment type for this study is 'In Situ Hybridization Assay'."

In [30]:
count_match_study_type = 0
for row, i in test.iterrows():
    
    if isinstance(i['Study_Type'], str) and i['Study_Type'].strip() != '':

        real_study = i['Study_Type']
        #print(real_study)
        response = i['Mistral_description_study_type']
        #print(response)
        clean_response = remove_special_characters(response)
        lower_clean_response = clean_response.lower()

        if real_study in lower_clean_response:
            print('The real study type is in the response.')
            test.at[row, 'Real_Study_Type_in_Response_descriptions'] = 'Yes'
            count_match_study_type +=1
        else:
            print('The real study type is NOT in the response.')
            test.at[row, 'Real_Study_Type_in_Response_descriptions'] = 'No'
    else:
        print('No real study type to check in the response.')
        test.at[row, 'Real_Study_Type_in_Response_descriptions'] = 'No species provided'

The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in the response.
The real study type is NOT in th

In [31]:
count_match_study_type

0

#### All dataframe - whole info

In [32]:
#test_wo_species = test.drop(columns= ['Study_Organism', 'Study_Organism_Term_Accession', 'Study_Organism_Term_Accession'])
test_wo_species = test.drop(columns= ['Study_Organism'])
test_wo_species

,Unnamed: 0,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Key_Words,Study_Organism_Term_Source_REF,Study_Organism_Term_Accession,...,Experiment_Description_word_count_clean,Study_Description_clean_lemmatize,Study_Description_clean_entities,Experiment_Description_clean_lemmatize,Experiment_Description_clean_entities,Description_combined,Mistral_description_combined_species,Real_Species_in_Response_descriptions,Mistral_description_study_type,Real_Study_Type_in_Response_descriptions
0,0,idr0092,a semi-automated organoid screening method dem...,compound library screen,efo,efo_0007553,intestinal organoids are an excellent model to...,organoids,ncbitaxon,10090,...,0,intestinal organoid excellent model study epit...,"[intestinal, model study, epithelial biology, ...",NaN,[],intestinal organoids are an excellent model to...,I don't have real-time access to databases or...,Yes,The experiment type for this study is 'In Sit...,No
1,1,idr0016,human u2os cells - compound cell-painting expe...,high content screen,efo,efo_0007550,phenotypic profiling attempts to summarize mul...,NaN,ncbitaxon,ncbitaxon_9606,...,0,phenotypic profiling attempt summarize multipa...,"[phenotypic profiling, multiparametric analysi...",NaN,[],phenotypic profiling attempts to summarize mul...,I don't know. The context provided does not s...,No,The experiment type for this study is 'In Sit...,No
2,2,idr0079,an image-based data-driven analysis of cellula...,microscopy assay,efo,efo_0002909,a data-driven analysis of cell morphology and ...,image analysis,ncbitaxon,7955,...,20,analysis cell morphology intracellular organiz...,"[morphology, intracellular, organization, zebr...",confocal imaging fixed sample zebrafish poster...,"[confocal imaging, posterior lateral line, pri...",a data-driven analysis of cell morphology and ...,"Based on the context provided, the scientific...",Yes,The experiment type for this study is 'In Sit...,No
3,3,idr0067,meiotic cellular rejuvenation is coupled to nu...,time-lapse imaging,omit,omit_0027490,production of healthy gametes in meiosis relie...,meiosis,ncbitaxon,4932,...,29,production healthy gamete meiosis rely quality...,"[production, healthy, meiosis, quality, distri...",fix cell fluorescence microscopy transmission ...,"[cell fluorescence microscopy, image, yeast, m...",production of healthy gametes in meiosis relie...,"Based on the context provided, the scientific...",No,The experiment type for this study is 'In Sit...,No
4,4,idr0134,reference bioimaging dataset to assess the phe...,histology,none,none,"here, we present a high-quality reference data...",phenotypes,ncbitaxon,264775,...,508,present reference dataset contain macroscopic ...,"[dataset, macroscopic, phenotypic property spe...",representative voucher specimen receive herbar...,"[specimen, herbaria, sample diplophyllum taxif...","here, we present a high-quality reference data...","Based on the context provided, the three scie...",No,The experiment type for this study is 'In Sit...,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,127,idr0023,nuclear pore scaffold structure analyzed by su...,protein localization,efo,go:0008104,much of life's essential molecular machinery c...,NaN,ncbitaxon,ncbitaxon_9606,...,11,much lifes essential molecular machinery consi...,"[lifes, molecular machinery, protein assembly,...",systematic immunolabelling complex monomeric e...,"[systematic immunolabelling, complex, monomeri...",much of life's essential molecular machinery c...,"Based on the context provided, the three scie...",Yes,The experiment type for this study is 'In Sit...,No
128,128,idr0038,ex vivo live cell tracking in kidney organoids...,time-lapse imaging,omit,omit_0027490,we have adapted the mouse kidney rudiment assa...,NaN,ncbitaxon,ncbitaxon_10090,...,161,adapt mouse kidney rudiment assay generate ren...,"[adapt, mouse kidney rudiment assay, renal or

### Clasical pathway

In [13]:
#pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.4/en_ner_bionlp13cg_md-0.5.4.tar.gz

In [14]:
#pip install spacy==3.7.4

In [15]:
# Load the spaCy model
## This model have the tags on it. One of them is 'ORG' so I choose it to extract the species.

nlp = spacy.load("en_ner_bionlp13cg_md") 
print(nlp.pipe_names) 

['tok2vec', 'tagger', 'attribute_ruler', 'lemmatizer', 'parser', 'ner']


/opt/conda/lib/python3.11/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [16]:
import time

In [17]:
# Function to extract species from the combined descriptions
# This function takes a text input, processes it with the spaCy model, and extracts named entities related to species.
# It returns a dictionary with the entity labels as keys and the corresponding entity texts as values.
# The function uses the spaCy model to identify named entities in the text and groups them by their entity labels.
# The output is a dictionary where the keys are the entity labels (e.g., 'ORG' for organisms) and the values are lists of entity texts that correspond to those labels.
# ------------------------------------------------------------  

def extract_species(text):
    doc= nlp(text)
    entities = [(ent.text, ent.label_) for ent in doc.ents]

    dictionary_entities = {}
    for k,v in entities:
        if v not in dictionary_entities:
            dictionary_entities[v]=[]
        dictionary_entities[v].append(k)
    
    return dictionary_entities

In [18]:
dfs = df_IDR
dfs = dfs.reset_index(drop=True)

In [19]:
def extract_entities(doc):
    out = {}
    for ent in doc.ents:
        out.setdefault(ent.label_, []).append(ent.text)
    return out

texts = dfs["Study_Description"].fillna("").astype(str).tolist()

results = []
for doc in nlp.pipe(texts):
    results.append(extract_entities(doc))

dfs["Entities_SpaCy"] = results   # keep as dicts
# or strings:
# dfs["Entities_SpaCy"] = [json.dumps(r) for r in results]


In [20]:
dfs

,Unnamed: 0,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Key_Words,Study_Organism,Study_Organism_Term_Source_REF,...,Study_Description_clean,Experiment_Description_clean,Study_Description_word_count_clean,Experiment_Description_word_count_clean,Study_Description_clean_lemmatize,Study_Description_clean_entities,Experiment_Description_clean_lemmatize,Experiment_Description_clean_entities,Description_combined,Entities_SpaCy
0,0,idr0092,a semi-automated organoid screening method dem...,compound library screen,efo,efo_0007553,intestinal organoids are an excellent model to...,organoids,mus musculus,ncbitaxon,...,"['intestinal', 'organoids', 'excellent', 'mode...",NaN,89,0,intestinal organoid excellent model study epit...,"[intestinal, model study, epithelial biology, ...",NaN,[],intestinal organoids are an excellent model to...,"{'CANCER': ['intestinal organoids', 'intestina..."
1,1,idr0016,human u2os cells - compound cell-painting expe...,high content screen,efo,efo_0007550,phenotypic profiling attempts to summarize mul...,NaN,homo sapiens,ncbitaxon,...,"['phenotypic', 'profiling', 'attempts', 'summa...",NaN,182,0,phenotypic profiling attempt summarize multipa...,"[phenotypic profiling, multiparametric analysi...",NaN,[],phenotypic profiling attempts to summarize mul...,"{'CELL': ['cellular', 'u2os cells', 'cell', 'c..."
2,2,idr0079,an image-based data-driven analysis of cellula...,microscopy assay,efo,efo_0002909,a data-driven analysis of cell morphology and ...,image analysis,danio rerio,ncbitaxon,...,"['analysis', 'cell', 'morphology', 'intracellu...","['confocal', 'imaging', 'fixed', 'samples', 'z...",60,20,analysis cell morphology intracellular organiz...,"[morphology, intracellular, organization, zebr...",confocal imaging fixed sample zebrafish poster...,"[confocal imaging, posterior lateral line, pri...",a data-driven analysis of cell morphology and ...,"{'CELL': ['cell', 'single-cell', 'cell', 'cell..."
3,3,idr0067,meiotic cellular rejuvenation is coupled to nu...,time-lapse imaging,omit,omit_0027490,production of healthy gametes in meiosis relie...,meiosis,saccharomyces cerevisiae\n,ncbitaxon,...,"['production', 'healthy', 'gametes', 'meiosis'...","['fixed', 'cell', 'fluorescence', 'microscopy'...",99,29,production healthy gamete meiosis rely quality...,"[production, healthy, meiosis, quality, distri...",fix cell fluorescence microscopy transmission ...,"[cell fluorescence microscopy, image, yeast, m...",production of healthy gametes in meiosis relie...,"{'CELLULAR_COMPONENT': ['nuclear', 'nuclear', ..."
4,4,idr0134,reference bioimaging dataset to assess the phe...,histology,none,none,"here, we present a high-quality reference data...",phenotypes,diplophyllum albicans,ncbitaxon,...,"['present', 'reference', 'dataset', 'containin...","['representative', 'voucher', 'specimens', 're...",32,508,present reference dataset contain macroscopic ...,"[dataset, macroscopic, phenotypic property spe...",representative voucher specimen receive herbar...,"[specimen, herbaria, sample diplophyllum taxif...","here, we present a high-quality reference data...",{}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,127,idr0023,nuclear pore scaffold structure analyzed by su...,protein localization,efo,go:0008104,much of life's essential molecular machinery c...,NaN,homo sapiens,ncbitaxon,...,"['much', 'lifes', 'essential', 'molecular', 'm...","['systematic', 'immunolabelling', 'complex', '...",75,11,much lifes essential molecular machinery consi...,"[lifes, molecular machinery, protein assembly,...",systematic immunolabelling complex monomeric e...,"[systematic immunolabelling, complex, monomeri...",much of life's essential molecular machinery c...,"{'CELLULAR_COMPONENT': ['nuclear', 'pores'], '..."
128,128,idr0038,ex vivo live cell tracking in kidney organoids...,time-lapse imaging,omit,omit_0027490,we have adapted the m

In [21]:
dfs["Species_SpaCy"] = dfs["Entities_SpaCy"].apply(
    lambda d: d.get("ORGANISM", "Not found") if isinstance(d, dict) else "Not found"
)

In [22]:
dfs

,Unnamed: 0,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Key_Words,Study_Organism,Study_Organism_Term_Source_REF,...,Experiment_Description_clean,Study_Description_word_count_clean,Experiment_Description_word_count_clean,Study_Description_clean_lemmatize,Study_Description_clean_entities,Experiment_Description_clean_lemmatize,Experiment_Description_clean_entities,Description_combined,Entities_SpaCy,Species_SpaCy
0,0,idr0092,a semi-automated organoid screening method dem...,compound library screen,efo,efo_0007553,intestinal organoids are an excellent model to...,organoids,mus musculus,ncbitaxon,...,NaN,89,0,intestinal organoid excellent model study epit...,"[intestinal, model study, epithelial biology, ...",NaN,[],intestinal organoids are an excellent model to...,"{'CANCER': ['intestinal organoids', 'intestina...",Not found
1,1,idr0016,human u2os cells - compound cell-painting expe...,high content screen,efo,efo_0007550,phenotypic profiling attempts to summarize mul...,NaN,homo sapiens,ncbitaxon,...,NaN,182,0,phenotypic profiling attempt summarize multipa...,"[phenotypic profiling, multiparametric analysi...",NaN,[],phenotypic profiling attempts to summarize mul...,"{'CELL': ['cellular', 'u2os cells', 'cell', 'c...",Not found
2,2,idr0079,an image-based data-driven analysis of cellula...,microscopy assay,efo,efo_0002909,a data-driven analysis of cell morphology and ...,image analysis,danio rerio,ncbitaxon,...,"['confocal', 'imaging', 'fixed', 'samples', 'z...",60,20,analysis cell morphology intracellular organiz...,"[morphology, intracellular, organization, zebr...",confocal imaging fixed sample zebrafish poster...,"[confocal imaging, posterior lateral line, pri...",a data-driven analysis of cell morphology and ...,"{'CELL': ['cell', 'single-cell', 'cell', 'cell...",[zebrafish posterior lateral line primordium]
3,3,idr0067,meiotic cellular rejuvenation is coupled to nu...,time-lapse imaging,omit,omit_0027490,production of healthy gametes in meiosis relie...,meiosis,saccharomyces cerevisiae\n,ncbitaxon,...,"['fixed', 'cell', 'fluorescence', 'microscopy'...",99,29,production healthy gamete meiosis rely quality...,"[production, healthy, meiosis, quality, distri...",fix cell fluorescence microscopy transmission ...,"[cell fluorescence microscopy, image, yeast, m...",production of healthy gametes in meiosis relie...,"{'CELLULAR_COMPONENT': ['nuclear', 'nuclear', ...",Not found
4,4,idr0134,reference bioimaging dataset to assess the phe...,histology,none,none,"here, we present a high-quality reference data...",phenotypes,diplophyllum albicans,ncbitaxon,...,"['representative', 'voucher', 'specimens', 're...",32,508,present reference dataset contain macroscopic ...,"[dataset, macroscopic, phenotypic property spe...",representative voucher specimen receive herbar...,"[specimen, herbaria, sample diplophyllum taxif...","here, we present a high-quality reference data...",{},Not found
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,127,idr0023,nuclear pore scaffold structure analyzed by su...,protein localization,efo,go:0008104,much of life's essential molecular machinery c...,NaN,homo sapiens,ncbitaxon,...,"['systematic', 'immunolabelling', 'complex', '...",75,11,much lifes essential molecular machinery consi...,"[lifes, molecular machinery, protein assembly,...",systematic immunolabelling complex monomeric e...,"[systematic immunolabelling, complex, monomeri...",much of life's essential molecular machinery c...,"{'CELLULAR_COMPONENT': ['nuclear', 'pores'], '...",Not found
128,128,idr0038,ex vivo live cell tracking in kidney organoids...,time-lapse imaging,omit,omit_0027490,we have adapted the mouse kidney rudiment assa...,NaN,mus musculus,ncbitaxon,...,"['adapted', 'mouse', 'kidney', 'rudiment', 'as...",79,161,adapt mouse kidney rudiment assay generate ren...,"[adapt, mouse kidney rudiment assay, renal org...",adapt

In [24]:
start_time = time.time()

start_time = time.time()

for i, item in dfs.iterrows():
    list_species_spacy = item['Species_SpaCy']
    list_species_scientific_name = []

    for x in list_species_spacy:
        values = x.split(' ')
        #print(values)
        for unique_value in values:
            #print(unique_value)
            
            url = f'https://www.ebi.ac.uk/ena/taxonomy/rest/any-name/{unique_value}'
            #print(url)

            response = requests.get(url)
        
            if response.status_code == 200:
                data = response.json()
                #print(data)
                if data:
                    scientific_name = data[0]['scientificName']
                    #print("found")

                    list_species_scientific_name.append(scientific_name)
                    print(f"Scientific name for {x}: {scientific_name}")
                else:
                    scientific_name = 'Not found'
                    print(f"Scientific name for {x}: {scientific_name}")

    dfs.at[i, 'Species_Scientific_Name_SciSpacy'] = str(list_species_scientific_name)
end_time = time.time()

print(f"Execution time: {end_time - start_time:.2f} seconds")

Scientific name for N: Not found
Scientific name for o: Not found
Scientific name for t: Not found
Scientific name for f: Not found
Scientific name for o: Not found
Scientific name for u: Not found
Scientific name for n: Not found
Scientific name for d: Not found
Scientific name for N: Not found
Scientific name for o: Not found
Scientific name for t: Not found
Scientific name for f: Not found
Scientific name for o: Not found
Scientific name for u: Not found
Scientific name for n: Not found
Scientific name for d: Not found
Scientific name for zebrafish posterior lateral line primordium: Danio rerio
Scientific name for zebrafish posterior lateral line primordium: Not found
Scientific name for zebrafish posterior lateral line primordium: Not found
Scientific name for zebrafish posterior lateral line primordium: Not found
Scientific name for zebrafish posterior lateral line primordium: Not found
Scientific name for N: Not found
Scientific name for o: Not found
Scientific name for t: Not fo

In [28]:
dfs

,Unnamed: 0,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Key_Words,Study_Organism,Study_Organism_Term_Source_REF,...,Experiment_Description_word_count_clean,Study_Description_clean_lemmatize,Study_Description_clean_entities,Experiment_Description_clean_lemmatize,Experiment_Description_clean_entities,Description_combined,Entities_SpaCy,Species_SpaCy,Species_Scientific_Name_SciSpacy,Specie_found_Scispacy
0,0,idr0092,a semi-automated organoid screening method dem...,compound library screen,efo,efo_0007553,intestinal organoids are an excellent model to...,organoids,mus musculus,ncbitaxon,...,0,intestinal organoid excellent model study epit...,"[intestinal, model study, epithelial biology, ...",NaN,[],intestinal organoids are an excellent model to...,"{'CANCER': ['intestinal organoids', 'intestina...",Not found,[],No
1,1,idr0016,human u2os cells - compound cell-painting expe...,high content screen,efo,efo_0007550,phenotypic profiling attempts to summarize mul...,NaN,homo sapiens,ncbitaxon,...,0,phenotypic profiling attempt summarize multipa...,"[phenotypic profiling, multiparametric analysi...",NaN,[],phenotypic profiling attempts to summarize mul...,"{'CELL': ['cellular', 'u2os cells', 'cell', 'c...",Not found,[],No
2,2,idr0079,an image-based data-driven analysis of cellula...,microscopy assay,efo,efo_0002909,a data-driven analysis of cell morphology and ...,image analysis,danio rerio,ncbitaxon,...,20,analysis cell morphology intracellular organiz...,"[morphology, intracellular, organization, zebr...",confocal imaging fixed sample zebrafish poster...,"[confocal imaging, posterior lateral line, pri...",a data-driven analysis of cell morphology and ...,"{'CELL': ['cell', 'single-cell', 'cell', 'cell...",[zebrafish posterior lateral line primordium],['Danio rerio'],Yes
3,3,idr0067,meiotic cellular rejuvenation is coupled to nu...,time-lapse imaging,omit,omit_0027490,production of healthy gametes in meiosis relie...,meiosis,saccharomyces cerevisiae\n,ncbitaxon,...,29,production healthy gamete meiosis rely quality...,"[production, healthy, meiosis, quality, distri...",fix cell fluorescence microscopy transmission ...,"[cell fluorescence microscopy, image, yeast, m...",production of healthy gametes in meiosis relie...,"{'CELLULAR_COMPONENT': ['nuclear', 'nuclear', ...",Not found,[],No
4,4,idr0134,reference bioimaging dataset to assess the phe...,histology,none,none,"here, we present a high-quality reference data...",phenotypes,diplophyllum albicans,ncbitaxon,...,508,present reference dataset contain macroscopic ...,"[dataset, macroscopic, phenotypic property spe...",representative voucher specimen receive herbar...,"[specimen, herbaria, sample diplophyllum taxif...","here, we present a high-quality reference data...",{},Not found,[],No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,127,idr0023,nuclear pore scaffold structure analyzed by su...,protein localization,efo,go:0008104,much of life's essential molecular machinery c...,NaN,homo sapiens,ncbitaxon,...,11,much lifes essential molecular machinery consi...,"[lifes, molecular machinery, protein assembly,...",systematic immunolabelling complex monomeric e...,"[systematic immunolabelling, complex, monomeri...",much of life's essential molecular machinery c...,"{'CELLULAR_COMPONENT': ['nuclear', 'pores'], '...",Not found,[],No
128,128,idr0038,ex vivo live cell tracking in kidney organoids...,time-lapse imaging,omit,omit_0027490,we have adapted the mouse kidney rudiment assa...,NaN,mus musculus,ncbitaxon,...,161,adapt mouse kidney rudiment assay generate ren...,"[adapt, mouse kidney rudiment assay, renal org...",adapt mouse kidney rudiment assay generate ren...,"[adapt, mouse kidney rudiment assay, renal org...",we have adapted the mouse kidney rudiment assa...,"{'ORGANISM': ['mouse kidney rudiment', 'e13.5 ...","[mouse kidney rudiment, e13.5 embryonic kidneys]"

In [29]:
def remove_special_characters(text):
    """
    This function removes special characters from the text.
    :param text: text
    :return: text without special characters
    """
    cleaned_string = re.sub(r'[?|$|.|!|@|#|%|^|&|*|(|)|-|_|+|=|;|:|,|<|>|/|{|}|[|]|~|`|\'|\"|\\]',r'', text)

    return cleaned_string

In [30]:
for i, item in dfs.iterrows():
    if isinstance(item['Study_Organism'], str) and item['Study_Organism'].strip() != '':
        real_specie = item['Study_Organism']
        scispacy_specie = item['Species_Scientific_Name_SciSpacy']
        clean_spacy_specie = remove_special_characters(scispacy_specie).lower()

        if real_specie in clean_spacy_specie:

            print('The real species is in SciSpacy')
            dfs.at[i, 'Specie_found_Scispacy'] = 'Yes'
        else:
            print('The real species is NOT in SciSpacy')
            dfs.at[i, 'Specie_found_Scispacy'] = 'No'
    else:
        print('No real species to check in SciSpacy')
        dfs.at[i, 'Specie_found_Scispacy'] = 'No species provided'

The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is in SciSpacy
The real species is NOT in SciSpacy
The real species is in SciSpacy
The real species is NOT in SciSpacy
The real species is in SciSpacy
The real species is in SciSpacy
The real species is in SciSpacy
The real species is in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is in SciSpacy
The real species is in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is NOT in SciSpacy
The real species is in SciSpacy
The 

In [31]:
dfs['Specie_found_Scispacy'].value_counts()

Specie_found_Scispacy
No                     80
Yes                    49
No species provided     3
Name: count, dtype: int64

In [1]:
import transformers

In [8]:
pip install "transformers==4.51.3"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 94.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 99.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.19.0
    Uninstalling huggingface_hub-1.19.0:
      Successfully uninstalled huggingface_hub-1.19.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.0
    Uninstalling transformers-5.12.0:
      Successfully uninstalled transformers-5.12.0
Note: you may need to restart the kernel to use updated packages.


In [32]:

# Use a pipeline for question answering with BioBERT
# This pipeline is used to answer questions based on the context provided.
# It uses the BioBERT model trained on the SQuAD dataset for question answering.
# The model is loaded with the device set to 'mps' for MacOS GPU support
# and is used to answer questions related to biomedical texts.
# The model is specifically designed for question answering tasks in the biomedical domain.
# ------------------------------------------------------------

from transformers import pipeline

pipe = pipeline("question-answering", model="dmis-lab/biobert-large-cased-v1.1-squad")

Device set to use cuda:0


In [34]:
from transformers import pipeline
import time

# Load the BioBERT model and create a pipeline for question answering
qa_pipeline_biobert = pipeline("question-answering", model="dmis-lab/biobert-large-cased-v1.1-squad")

start_time = time.time()

for i, item in dfs.iterrows():
    specie_found = item['Specie_found_Scispacy']

    if specie_found.startswith("No"):
        # Use the BioBERT pipeline to answer the question
        text = item['Description_combined']
        question = 'what species was used in the experiment?'

        result = qa_pipeline_biobert(question=question, context=text)
        print(f"Row {i}: {result['answer']}")
        dfs.at[i, 'Species_BioBERT'] = result['answer']

    else:
        print(f"Row {i}: The species was already found in the previous steps.")

end_time = time.time()

print(f"Execution time: {end_time - start_time:.2f} seconds")

Device set to use cuda:0


Row 0: small intestinal crypts were isolated
Row 1: u2os
Row 2: The species was already found in the previous steps.
Row 3: budding yeast


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Row 4: diplophyllum taxifolium
Row 5: The species was already found in the previous steps.
Row 6: hela
Row 7: The species was already found in the previous steps.
Row 8: yeast
Row 9: The species was already found in the previous steps.
Row 10: The species was already found in the previous steps.
Row 11: The species was already found in the previous steps.
Row 12: The species was already found in the previous steps.
Row 13: murine
Row 14: bovine
Row 15: mda-mb-231
Row 16: The species was already found in the previous steps.
Row 17: mammalian cells
Row 18: hela
Row 19: The species was already found in the previous steps.
Row 20: yeast
Row 21: escherichia coli
Row 22: The species was already found in the previous steps.
Row 23: The species was already found in the previous steps.
Row 24: tribolium castaneum
Row 25: arabidopsis thaliana
Row 26: fetal bovine
Row 27: arabidopsis
Row 28: The species was already found in the previous steps.
Row 29: mouse
Row 30: plankton
Row 31: The species wa

In [35]:
# Creating a new column with the scientific names of the species extracted from the Species_BioBERT column
# This column will be used to store the scientific names of the species extracted from the Species_BioBERT column.
# It uses the EBI taxonomy REST API to get the scientific names of the species.
# It iterates over the Species_BioBERT column, splits the species names, and queries the EBI taxonomy REST API for each species name.
# The scientific names are stored in a list and added to the Species_Scientific_Name_BioBERT column.
# ------------------------------------------------------------

for i, item in dfs.iterrows():
    list_species = item['Species_BioBERT']
    #list_species_scientific_name = []

    if isinstance(list_species, str):
        list_species = list_species.split(', ')
        #print(list_species)

        if list_species != []:
        
            #print(list_species)
            for x in list_species:
                #print(x)

                url = f'https://www.ebi.ac.uk/ena/taxonomy/rest/any-name/{x}'
                    #print(url)

                response = requests.get(url)
                    #print(response.status_code)
                if response.status_code == 200:
                    data = response.json()
                    #print(data)
                    if data:
                        scientific_name = data[0]['scientificName']

                        #list_species_scientific_name.append(scientific_name)
                        print(f"Scientific name for {x}: {scientific_name}")

            dfs.at[i, 'Species_Scientific_Name_BioBERT'] = str(scientific_name)
        else:
            dfs.at[i, 'Species_Scientific_Name_BioBERT'] = 'Not found'
        

Scientific name for diplophyllum taxifolium: Diplophyllum taxifolium
Scientific name for bovine: Bos taurus
Scientific name for escherichia coli: Escherichia coli
Scientific name for tribolium castaneum: Tribolium castaneum
Scientific name for arabidopsis thaliana: Arabidopsis thaliana
Scientific name for arabidopsis: Arabidopsis
Scientific name for mouse: Mus musculus
Scientific name for arabidopsis: Arabidopsis
Scientific name for fission yeast: Schizosaccharomyces pombe
Scientific name for mouse: Mus musculus
Scientific name for sea urchin: Echinoidea
Scientific name for mice: Mus sp.
Scientific name for horse: Equus caballus
Scientific name for goat: Capra hircus
Scientific name for drosophila melanogaster: Drosophila melanogaster
Scientific name for honey bees: Apinae
Scientific name for mouse: Mus musculus
Scientific name for mice: Mus sp.
Scientific name for e. coli: Escherichia coli
Scientific name for homo sapiens: Homo sapiens
Scientific name for bacteria: Bacteria
Scientific

In [36]:
dfs['Species_Scientific_Name_BioBERT'] = dfs['Species_Scientific_Name_BioBERT'].astype(str)

In [37]:
dfs

,Unnamed: 0,Comment[IDR_Study_Accession],Study_Title,Study_Type,Study_Type_Term_Source_REF,Study_Type_Term_Accession,Study_Description,Study_Key_Words,Study_Organism,Study_Organism_Term_Source_REF,...,Study_Description_clean_entities,Experiment_Description_clean_lemmatize,Experiment_Description_clean_entities,Description_combined,Entities_SpaCy,Species_SpaCy,Species_Scientific_Name_SciSpacy,Specie_found_Scispacy,Species_BioBERT,Species_Scientific_Name_BioBERT
0,0,idr0092,a semi-automated organoid screening method dem...,compound library screen,efo,efo_0007553,intestinal organoids are an excellent model to...,organoids,mus musculus,ncbitaxon,...,"[intestinal, model study, epithelial biology, ...",NaN,[],intestinal organoids are an excellent model to...,"{'CANCER': ['intestinal organoids', 'intestina...",Not found,[],No,small intestinal crypts were isolated,Not found
1,1,idr0016,human u2os cells - compound cell-painting expe...,high content screen,efo,efo_0007550,phenotypic profiling attempts to summarize mul...,NaN,homo sapiens,ncbitaxon,...,"[phenotypic profiling, multiparametric analysi...",NaN,[],phenotypic profiling attempts to summarize mul...,"{'CELL': ['cellular', 'u2os cells', 'cell', 'c...",Not found,[],No,u2os,Not found
2,2,idr0079,an image-based data-driven analysis of cellula...,microscopy assay,efo,efo_0002909,a data-driven analysis of cell morphology and ...,image analysis,danio rerio,ncbitaxon,...,"[morphology, intracellular, organization, zebr...",confocal imaging fixed sample zebrafish poster...,"[confocal imaging, posterior lateral line, pri...",a data-driven analysis of cell morphology and ...,"{'CELL': ['cell', 'single-cell', 'cell', 'cell...",[zebrafish posterior lateral line primordium],['Danio rerio'],Yes,NaN,NaN
3,3,idr0067,meiotic cellular rejuvenation is coupled to nu...,time-lapse imaging,omit,omit_0027490,production of healthy gametes in meiosis relie...,meiosis,saccharomyces cerevisiae\n,ncbitaxon,...,"[production, healthy, meiosis, quality, distri...",fix cell fluorescence microscopy transmission ...,"[cell fluorescence microscopy, image, yeast, m...",production of healthy gametes in meiosis relie...,"{'CELLULAR_COMPONENT': ['nuclear', 'nuclear', ...",Not found,[],No,budding yeast,Not found
4,4,idr0134,reference bioimaging dataset to assess the phe...,histology,none,none,"here, we present a high-quality reference data...",phenotypes,diplophyllum albicans,ncbitaxon,...,"[dataset, macroscopic, phenotypic property spe...",representative voucher specimen receive herbar...,"[specimen, herbaria, sample diplophyllum taxif...","here, we present a high-quality reference data...",{},Not found,[],No,diplophyllum taxifolium,Diplophyllum taxifolium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,127,idr0023,nuclear pore scaffold structure analyzed by su...,protein localization,efo,go:0008104,much of life's essential molecular machinery c...,NaN,homo sapiens,ncbitaxon,...,"[lifes, molecular machinery, protein assembly,...",systematic immunolabelling complex monomeric e...,"[systematic immunolabelling, complex, monomeri...",much of life's essential molecular machinery c...,"{'CELLULAR_COMPONENT': ['nuclear', 'pores'], '...",Not found,[],No,fetal bovine,Homo sapiens
128,128,idr0038,ex vivo live cell tracking in kidney organoids...,time-lapse imaging,omit,omit_0027490,we have adapted the mouse kidney rudiment assa...,NaN,mus musculus,ncbitaxon,...,"[adapt, mouse kidney rudiment assay, renal org...",adapt mouse kidney rudiment assay generate ren...,"[adapt, mouse kidney rudiment assay, renal org...",we have adapted the mouse kidney rudiment assa...,"{'ORGANISM': ['mouse kidney rudiment', 'e13.5 ...","[mouse kidney rudiment, e13.5 embryonic kidneys]",['Mus musculus'],Yes,NaN,NaN
129,129,idr0138,integration of spatial and single-cell transcr...,seqfish,efo,efo_0008991,molecular profiling of single cells has advanc...,seqfish,mus musculus,ncbitaxon,...,"[molecular profiling, 

In [ ]:
dfs['Species_Scientific_Name_BioBERT'][4] == 'Homo sapiens'

In [38]:
for i, item in dfs.iterrows():
    specie_found = item['Specie_found_Scispacy']

    if specie_found.startswith("No"):

        if isinstance(item['Study_Organism'], str) and item['Study_Organism'].strip() != '':
            real_specie = item['Study_Organism'][0]
            bert_specie = item['Species_Scientific_Name_BioBERT'][0]
            clean_biobert_specie = remove_special_characters(bert_specie).lower()

            if real_specie in clean_biobert_specie:

                print('The real species is in the bert')
                dfs.at[i, 'Specie_found_biobert'] = 'Yes'
            else:
                print('The real species is NOT in the response.')
                dfs.at[i, 'Specie_found_biobert'] = 'No'
        else:
            print('No real species to check in the response.')
            dfs.at[i, 'Specie_found_biobert'] = 'No species provided'
  
            

The real species is NOT in the response.
The real species is NOT in the response.
The real species is NOT in the response.
The real species is in the bert
The real species is NOT in the response.
The real species is NOT in the response.
The real species is NOT in the response.
The real species is NOT in the response.
The real species is NOT in the response.
The real species is NOT in the response.
The real species is NOT in the response.
The real species is NOT in the response.
The real species is in the bert
The real species is in the bert
The real species is in the bert
The real species is NOT in the response.
The real species is in the bert
The real species is in the bert
The real species is in the bert
The real species is NOT in the response.
The real species is in the bert
The real species is in the bert
The real species is in the bert
The real species is NOT in the response.
The real species is NOT in the response.
The real species is NOT in the response.
The real species is NOT 

In [39]:
dfs['Specie_found_biobert'].value_counts()

Specie_found_biobert
No                     46
Yes                    34
No species provided     3
Name: count, dtype: int64

In [ ]:
type(dfs['Specie_found_biobert'][2])

In [ ]:
dfs

In [43]:
for row, i in dfs.iterrows():
    value_biobert = str(i['Specie_found_biobert'])
    
    if value_biobert.startswith("No"):
    
        message = [
        {"role": "system", "content": i["Description_combined"]},
        {"role": "user", "content": "Can you tell me which scientific species was used in this study?"},
        ]

        outputs = pipe(message, max_new_tokens = 256)
        response = outputs[0]["generated_text"][-1]["content"]
        clean_response = remove_special_characters(response).lower()
        
        dfs.at[row, 'Species_Mistral'] = clean_response
        print(f"Processed row {row} for species extraction.")

        if isinstance(i['Study_Organism'], str) and i['Study_Organism'].strip() != '':

            real_specie = i['Study_Organism'][0]

            if real_specie in clean_response:
                print('The real species is in the response.')
                dfs.at[row, 'Species_found_Mistral'] = 'Yes'
            else:
                print('The real species is NOT in the response.')
                dfs.at[row, 'Species_found_Mistral'] = 'No'
        else:
            print('No real species to check in the response.')
            dfs.at[row, 'Species_found_Mistral'] = 'No species provided'

end_time = time.time()

print(f"Execution time: {end_time - start_time:.2f} seconds")    

Processed row 0 for species extraction.
The real species is in the response.
Processed row 1 for species extraction.
The real species is in the response.
Processed row 3 for species extraction.
The real species is in the response.


KeyboardInterrupt: 

In [ ]:
dfs['Species_found_Mistral'].value_counts()

In [41]:
model_id = "meta-llama/Llama-3.2-3B-Instruct"
pipe = pipeline("text-generation",
                model = model_id,
                torch_dtype = torch.bfloat16,
                device_map="auto",
                pad_token_id=128001 
               )

Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████████████████████| 2/2 [00:04<00:00,  2.02s/it]
Device set to use cuda:0


In [45]:
start_time = time.time()
for row, i in dfs.iterrows():
    value_biobert = str(i['Specie_found_biobert'])

    if value_biobert.startswith("No"):
    
        message = [
        {"role": "system", "content": i["Description_combined"]},
        {"role": "user", "content": "Can you tell me which scientific species was used in this study?"},
        ]

        outputs = pipe(message, max_new_tokens = 256)
        response = outputs[0]["generated_text"][-1]["content"]
        clean_response = remove_special_characters(response).lower()
        
        dfs.at[row, 'Species_Llama'] = clean_response
        print(f"Processed row {row} for species extraction.")

        
        if isinstance(i['Study_Organism'], str) and i['Study_Organism'].strip() != '':
            real_specie = i['Study_Organism'].strip()  # remove leading and trailing whitespace
        
            if real_specie in clean_response:
                print('The real species is in the response.')
                dfs.at[row, 'Species_found_Llama'] = 'Yes'
            else:
                print('The real species is NOT in the response.')
                dfs.at[row, 'Species_found_Llama'] = 'No'
        else:
            print('No real species to check in the response.')
            dfs.at[row, 'Species_found_Llama'] = 'No species provided'


end_time = time.time()

print(f"Execution time: {end_time - start_time:.2f} seconds")

Processed row 0 for species extraction.
The real species is in the response.
Processed row 1 for species extraction.
The real species is NOT in the response.
Processed row 3 for species extraction.
The real species is in the response.
Processed row 6 for species extraction.
The real species is in the response.
Processed row 8 for species extraction.
The real species is in the response.
Processed row 13 for species extraction.
The real species is NOT in the response.
Processed row 14 for species extraction.
The real species is in the response.
Processed row 15 for species extraction.
The real species is NOT in the response.
Processed row 17 for species extraction.
The real species is NOT in the response.
Processed row 18 for species extraction.
The real species is NOT in the response.
Processed row 20 for species extraction.
The real species is in the response.
Processed row 26 for species extraction.
The real species is in the response.
Processed row 32 for species extraction.
The real

In [46]:
dfs['Species_found_Llama'].value_counts()

Species_found_Llama
No                     25
Yes                    21
No species provided     3
Name: count, dtype: int64

In [ ]:
#dfs = dfs.rename(columns = {"Species_found_Llama":"Species_found_Mistral" })

In [ ]:
for row, i in dfs.iterrows():
    value_scipacy = str(i['Specie_found_biobert'])
    print(value_scipacy)
    
    value_biobert = str(i['Specie_found_Scispacy'])
    print(value_biobert)
    
    value_mistral = str(i['Species_found_Mistral'])
    print(value_mistral)
    
    value_llama = str(i['Species_found_Llama'])
    print(value_llama)

    if value_scipacy.startswith("Yes") or value_biobert.startswith("Yes") or value_lama.startswith("Yes"):
        print(f"Row {row}: Species found in one of the methods.")
        dfs.at[row, 'Species_found'] = 'Yes'

    else:
        print(f"Row {row}: Species not found in any method.")
        dfs.at[row, 'Species_found'] = 'No'

In [ ]:
dfs['Species_found'].value_counts()

In [ ]:
count_spacy = 0
count_biobert = 0
count_mistral = 0 
count_llama = 0

for row, i in dfs.iterrows():
    value_scipacy = str(i['Specie_found_Scispacy'])
    if value_scipacy.startswith("Yes") :
        count_spacy +=1
        
    value_biobert = str(i['Specie_found_biobert'])
    if value_biobert.startswith("Yes") :
        count_biobert +=1
    
    value_mistral = str(i['Species_found_Mistral'])
    if value_mistral.startswith("Yes") :
        count_mistral +=1
    
    value_llama = str(i['Species_found_Llama'])
    if value_llama.startswith("Yes") :
        count_llama +=1

print(f'Count spacy: {count_spacy}; Count biobert: {count_biobert}; Count Mistral: {count_mistral}; Count Llama {count_llama}')


In [ ]:
for row, i in dfs.iterrows():
    value_scipacy = str(i['Specie_found_biobert'])
    value_biobert = i['Specie_found_Scispacy']
    value_lama = i['Species_found_Llama']
    value_mistral = i['Species_found_Mistral']

    if value_scipacy.startswith("Yes") :
        print(f"Row {row}: Species found in SciSpacy, but not in BioBERT.") ## I am not sure about this